In [ ]:
!uv pip install -U transformers datasets accelerate torch scikit-learn pandas numpy

### Fix numpy version conflict

The previous `!uv pip install` command might have installed a `numpy` version (2.x) that is incompatible with `scipy` or `scikit-learn`, leading to the `AttributeError`. To fix this, we will explicitly install `numpy==1.26.4` and ensure `scipy` and `scikit-learn` are reinstalled to be compatible with this `numpy` version.

In [ ]:
# Uninstall current numpy, scipy, and scikit-learn to prepare for specific version installation
!uv pip uninstall numpy scipy scikit-learn

# Install numpy==1.26.4 and reinstall scipy and scikit-learn to ensure compatibility
# uv will resolve compatible versions for scipy and scikit-learn with numpy==1.26.4
!uv pip install numpy==1.26.4 scipy scikit-learn

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
import pandas as pd
import numpy as np

df=pd.read_csv("final_fake_real_dataset.csv")

In [ ]:
from sklearn.model_selection import train_test_split

# Load CSV
df = pd.read_csv("final_fake_real_dataset.csv")

# Check columns
print(df.head())
print(df.columns)

# Example: keep only text + labels columns
# Replace names if needed
# df = df[["text", "labels"]]

# Train test split
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print(train_df.shape)
print(test_df.shape)

In [ ]:
# First, uninstall to ensure a clean slate
!uv pip uninstall torch torchvision torchaudio transformers accelerate
!pip cache purge

# Install torch, torchvision, and torchaudio from the official PyTorch index
# This helps ensure compatibility between the libraries and their C++ extensions.
# Assuming CUDA 12.1, change cu121 to your specific CUDA version if different.
!uv pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Reinstall other necessary packages
!uv pip install transformers datasets accelerate scikit-learn pandas numpy

In [ ]:
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)
print(torch.cuda.is_available())

In [ ]:
# What your SimpleTransformers code is internally doing
# Rewritten using Hugging Face Transformers (recommended)

# Install if needed:
# pip install transformers datasets torch scikit-learn pandas

import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ---------------------------------------------------
# Expected input format:
# train_df columns = ["text", "labels"]
# test_df columns  = ["text", "labels"]
# labels: 0 = Fake, 1 = Real (example)
# ---------------------------------------------------

# Example:
# train_df = pd.DataFrame({
#     "text": ["news one", "news two"],
#     "labels": [0,1]
# })

# Rename columns to match expected format
train_df = train_df.rename(columns={"article": "text", "label": "labels"})
test_df = test_df.rename(columns={"article": "text", "label": "labels"})

# Convert pandas -> Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset  = Dataset.from_pandas(test_df)

# Load tokenizer (RoBERTa)
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

# Tokenization
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Keep only needed columns
train_dataset = train_dataset.remove_columns(
    [col for col in train_dataset.column_names if col not in ["input_ids", "attention_mask", "labels"]]
)

test_dataset = test_dataset.remove_columns(
    [col for col in test_dataset.column_names if col not in ["input_ids", "attention_mask", "labels"]]
)

train_dataset.set_format("torch")
test_dataset.set_format("torch")

# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

# Metrics (what eval_model gives)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

# Training settings
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="none"
)

# Trainer = internally what simpletransformers uses conceptually
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# Train model
trainer.train()

# Evaluate model
result = trainer.evaluate()

print(result)


In [ ]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# ---------------------------------------------------
# 1. LOAD DATASET
# ---------------------------------------------------
df = pd.read_csv("final_fake_real_dataset.csv")

print("Columns:", df.columns)

# Rename columns automatically
rename_map = {
    "label": "labels",
    "Label": "labels",
    "class": "labels",
    "target": "labels",
    "news": "text",
    "content": "text",
    "article": "text",
    "headline": "text",
    "title": "text"
}

df = df.rename(columns=rename_map)

# Keep only required columns
df = df[["text", "labels"]]

df = df.dropna()
df["labels"] = df["labels"].astype(int)

# ---------------------------------------------------
# 2. TRAIN TEST SPLIT
# ---------------------------------------------------
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["labels"]
)

# ---------------------------------------------------
# 3. METRICS FUNCTION
# ---------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# ---------------------------------------------------
# 4. MODELS TO COMPARE
# ---------------------------------------------------
models = {
    "DistilBERT": "distilbert-base-uncased",
    "BERT": "bert-base-uncased",
    "RoBERTa": "roberta-base",
    "ALBERT": "albert-base-v2"
}

results = []

# ---------------------------------------------------
# 5. LOOP THROUGH MODELS
# ---------------------------------------------------
for model_name, model_checkpoint in models.items():

    print(f"\n========== Training {model_name} ==========\n")

    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

    train_dataset = Dataset.from_pandas(train_df)
    test_dataset = Dataset.from_pandas(test_df)

    def tokenize_function(example):
        return tokenizer(
            example["text"],
            truncation=True,
            padding="max_length",
            max_length=256
        )

    train_dataset = train_dataset.map(tokenize_function, batched=True)
    test_dataset = test_dataset.map(tokenize_function, batched=True)

    train_dataset = train_dataset.remove_columns(
        [col for col in train_dataset.column_names if col not in ["input_ids", "attention_mask", "labels"]]
    )

    test_dataset = test_dataset.remove_columns(
        [col for col in test_dataset.column_names if col not in ["input_ids", "attention_mask", "labels"]]
    )

    train_dataset.set_format("torch")
    test_dataset.set_format("torch")

    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=2
    )

    training_args = TrainingArguments(
        output_dir=f"./{model_name}_results",
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=100,

        num_train_epochs=2,

        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,

        learning_rate=2e-5,
        weight_decay=0.01,

        fp16=torch.cuda.is_available(),

        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()

    # Save the trained model
    trainer.save_model(f"./saved_{model_name}")

    result = trainer.evaluate()

    results.append({
        "Model": model_name,
        "Accuracy": result["eval_accuracy"],
        "Precision": result["eval_precision"],
        "Recall": result["eval_recall"],
        "F1 Score": result["eval_f1"]
    })

# ---------------------------------------------------
# 6. FINAL COMPARISON TABLE
# ---------------------------------------------------
results_df = pd.DataFrame(results)

print("\n================ FINAL RESULTS ================\n")
print(results_df.sort_values(by="Accuracy", ascending=False))

In [ ]:
# ======================================================
# SAVE ALL MODEL RESULTS TO CSV + EXCEL + TXT
# Add this AFTER your benchmarking code finishes
# ======================================================

import pandas as pd

# results list already created in your benchmark code
# Convert to DataFrame
results_df = pd.DataFrame(results)

# Sort by Accuracy
results_df = results_df.sort_values(by="Accuracy", ascending=False)

# ------------------------------------------
# 1. Save CSV
# ------------------------------------------
results_df.to_csv("model_comparison_results.csv", index=False)

# ------------------------------------------
# 2. Save Excel
# ------------------------------------------
results_df.to_excel("model_comparison_results.xlsx", index=False)

# ------------------------------------------
# 3. Save Text File
# ------------------------------------------
with open("model_comparison_results.txt", "w") as f:
    f.write(results_df.to_string(index=False))

# ------------------------------------------
# 4. Print Final Table
# ------------------------------------------
print(results_df)

print("\nFiles Saved Successfully:")
print("1. model_comparison_results.csv")
print("2. model_comparison_results.xlsx")
print("3. model_comparison_results.txt")

In [ ]:
# ======================================================
# MULTI-MODEL BENCHMARK FOR FAKE NEWS DETECTION
# Train on one dataset and test on a NEW dataset
# Models:
# 1. DistilBERT
# 2. BERT-base
# 3. RoBERTa-base
# 4. ALBERT-base
# ======================================================

# pip install -U transformers datasets accelerate torch scikit-learn pandas numpy

import os
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)


# ======================================================
# 7. TEST ON NEW DATASET
# ======================================================

new_df = pd.read_csv("new_test_dataset.csv")

print("\nNew test columns:", new_df.columns)

new_rename_map = {
    "label": "labels",
    "Label": "labels",
    "class": "labels",
    "target": "labels",
    "news": "text",
    "content": "text",
    "article": "text",
    "headline": "text",
    "title": "text"
}

new_df = new_df.rename(columns=new_rename_map)

# Keep only text and labels if labels exist
if "labels" in new_df.columns:
    new_df = new_df[["text", "labels"]]
    new_df = new_df.dropna()
    new_df["labels"] = new_df["labels"].astype(int)
else:
    new_df = new_df[["text"]].dropna()

new_results = []

for model_name, model_checkpoint in models.items():
    print(f"\n========== Testing {model_name} on NEW data ==========\n")

    save_path = f"./saved_{model_name}"

    tokenizer = AutoTokenizer.from_pretrained(save_path)
    model = AutoModelForSequenceClassification.from_pretrained(save_path)

    new_dataset = Dataset.from_pandas(new_df.reset_index(drop=True))

    def tokenize_new(example):
        return tokenizer(
            example["text"],
            truncation=True,
            padding="max_length",
            max_length=256
        )

    new_dataset = new_dataset.map(tokenize_new, batched=True)

    remove_cols_new = [c for c in new_dataset.column_names if c not in ["input_ids", "attention_mask", "labels"] and c != "text"]
    if "text" in new_dataset.column_names:
        remove_cols_new.append("text")

    # Remove only columns that exist
    remove_cols_new = [c for c in remove_cols_new if c in new_dataset.column_names]
    new_dataset = new_dataset.remove_columns(remove_cols_new)

    new_dataset.set_format("torch")

    new_trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=f"./temp_{model_name}",
            per_device_eval_batch_size=16,
            report_to="none"
        )
    )

    preds_output = new_trainer.predict(new_dataset)
    logits = preds_output.predictions
    preds = np.argmax(logits, axis=1)

    if "labels" in new_df.columns:
        labels = np.array(new_df["labels"])

        acc = accuracy_score(labels, preds)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, preds, average="binary", zero_division=0
        )

        print(f"Accuracy : {acc:.4f}")
        print(f"Precision : {precision:.4f}")
        print(f"Recall    : {recall:.4f}")
        print(f"F1 Score  : {f1:.4f}")
        print("\nClassification Report:")
        print(classification_report(labels, preds, zero_division=0))

        new_results.append({
            "Model": model_name,
            "Accuracy": acc,
            "Precision": precision,
            "Recall": recall,
            "F1 Score": f1
        })
    else:
        print("No labels found in new test data. Showing predictions only.")
        print("Predictions:", preds[:20])

# ---------------------------------------------------
# 8. NEW DATA COMPARISON TABLE
# ---------------------------------------------------
if new_results:
    new_results_df = pd.DataFrame(new_results)
    print("\n=========== RESULTS ON NEW TEST DATA ===========\n")
    print(new_results_df.sort_values(by="Accuracy", ascending=False))

In [ ]:
print("Actual labels:")
print(pd.Series(labels).value_counts())

print("Predicted labels:")
print(pd.Series(preds).value_counts())